## 1. Verificación de la carga
Confirmamos que los 115 días del periodo (28 abr – 20 ago 2026) se subieron correctamente al volume.

In [0]:
base = "/Volumes/workspace/default/applebees/Applebees_Challenge"
len(dbutils.fs.ls(f"{base}/facts/checks/brand=rmh"))

## 2. Formato de los archivos
Inspeccionamos un archivo: cada checks.json es una lista de tickets, y cada ticket trae listas anidadas de productos (ItemsSold), pagos (Payments), impuestos (Tax), descuentos (Discounts) y cancelaciones (Voids)

In [0]:
print(dbutils.fs.head(f"{base}/facts/checks/brand=rmh/dt=2026-04-28/checks.json", 2000))

## 3. Lectura de los datos
Inferimos el schema de un solo día para no leer los 10 GB dos veces, y forzamos ServiceCharge a decimal para no perder registros. También cargamos las dimensiones: sucursales, canales de venta, productos y empleados.

In [0]:
from pyspark.sql.types import DoubleType, StructField, StructType

muestra = spark.read.option("multiLine", True).json(f"{base}/facts/checks/brand=rmh/dt=2026-04-28/checks.json")
esquema = StructType([StructField(f.name, DoubleType(), True) if f.name == "ServiceCharge" else f
                      for f in muestra.schema.fields])
checks = spark.read.option("multiLine", True).schema(esquema).json(f"{base}/facts/checks")

locations = spark.read.option("multiLine", True).json(f"{base}/dimensions/locations/locations.json")
rcs = spark.read.option("multiLine", True).json(f"{base}/dimensions/revenue_centers/revenueCenters.json")
items = spark.read.option("multiLine", True).json(f"{base}/dimensions/items/items.json")
employees = spark.read.option("multiLine", True).json(f"{base}/dimensions/employees/employees.json")

checks.printSchema()

## 4. Guardado como tablas Delta
Guardamos todo como tablas Delta para que las consultas posteriores sean rápidas y no dependan de releer los JSON.

In [0]:
checks.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable("workspace.default.checks")
for nombre, df in [("locations", locations), ("revenue_centers", rcs), ("items", items), ("employees", employees)]:
    df.write.mode("overwrite").saveAsTable(f"workspace.default.{nombre}")

## 5. Validación de la lectura
Verificamos que ningún ticket quedó sin Id o sin sucursal. Resultado: 0 registros perdidos.

In [0]:
display(spark.table("workspace.default.checks").filter("Id IS NULL OR LocationId IS NULL").count())

## 6. Tabla resumen por ticket
Creamos checks_resumen, con un renglón por ticket y métricas calculadas a partir de las listas anidadas: venta, impuestos, descuentos, cancelaciones, pagos, propinas, número de platillos, hora y duración de la mesa.

In [0]:
from pyspark.sql import functions as F

checks = spark.table("workspace.default.checks")
suma = lambda arr, campo: F.aggregate(F.coalesce(arr, F.array()), F.lit(0.0), lambda acc, x: acc + F.coalesce(x[campo].cast("double"), F.lit(0.0)))

c = (checks
  .withColumn("open_ts", F.to_timestamp("OpenTime"))
  .withColumn("close_ts", F.to_timestamp("CloseTime"))
  .withColumn("ventas", suma("ItemsSold", "SoldPrice"))
  .withColumn("bruto", suma("ItemsSold", "GrossPrice"))
  .withColumn("impuestos", suma("Tax", "Amount"))
  .withColumn("descuentos", suma("Discounts", "Amount"))
  .withColumn("cancelado", suma("Voids", "Amount"))
  .withColumn("pagado", suma("Payments", "Amount"))
  .withColumn("propina", suma("Payments", "Gratuity"))
  .withColumn("n_items", F.size(F.filter("ItemsSold", lambda x: x.ParentSequence == 0)))
  .withColumn("hora", F.hour("open_ts"))
  .withColumn("dia_semana", F.date_format("dt", "E"))
  .withColumn("min_mesa", (F.unix_timestamp("close_ts") - F.unix_timestamp("open_ts")) / 60)
  .drop("ItemsSold", "Payments", "Tax", "Discounts", "Voids"))

c.write.mode("overwrite").saveAsTable("workspace.default.checks_resumen")

## 7. Calidad de datos
Contamos duplicados, tickets con venta cero o negativa, tickets sin comensales, horarios inconsistentes y tickets donde la venta no cuadra con lo pagado.


In [0]:
c = spark.table("workspace.default.checks_resumen")
display(c.select(
    F.count("*").alias("tickets"),
    F.countDistinct("dt").alias("dias"),
    F.countDistinct("LocationId").alias("ubicaciones"),
    (F.count("*") - F.countDistinct("LocationId", "BusinessDate", "Id")).alias("duplicados"),
    F.sum((F.col("ventas") <= 0).cast("int")).alias("venta_cero_o_neg"),
    F.sum((F.col("TrafficCount") == 0).cast("int")).alias("sin_comensales"),
    F.sum((F.col("min_mesa") < 0).cast("int")).alias("cierre_antes_apertura"),
    F.sum((F.abs(F.col("ventas") + F.col("impuestos") + F.col("ServiceCharge") - F.col("pagado")) > 0.05).cast("int")).alias("no_cuadra_pago")
))

## 8. Dimensiones
Revisamos el catálogo de sucursales (185, de las cuales 127 tienen ventas en el periodo) y de canales de venta (23).

In [0]:
display(spark.table("workspace.default.locations"))
display(spark.table("workspace.default.revenue_centers"))

## 9. Cuadre de ventas contra pagos
Probamos varias fórmulas para reconstruir lo pagado. La mejor es venta − descuentos + impuestos + cargo de servicio, que cuadra en 52% de los tickets. Queda como pendiente de calidad de datos.

In [0]:
r = spark.table("workspace.default.checks")
s = lambda arr, f: F.aggregate(F.coalesce(arr, F.array()), F.lit(0.0), lambda a, x: a + F.coalesce(f(x), F.lit(0.0)))
d = r.select(
    s("ItemsSold", lambda x: x.SoldPrice).alias("v1"),
    s("ItemsSold", lambda x: x.SoldPrice * x.Quantity).alias("v2"),
    s("Tax", lambda x: x.Amount).alias("tax"),
    s("Discounts", lambda x: x.Amount).alias("desc"),
    s("Payments", lambda x: x.Amount).alias("pag"),
    s("Payments", lambda x: x.Gratuity).alias("prop"),
    F.coalesce("ServiceCharge", F.lit(0.0)).alias("sc"),
    F.size("Payments").alias("n_pagos"))
cuadra = lambda e: F.round(F.avg((F.abs(e) <= 0.05).cast("int")) * 100, 1)
display(d.select(
    cuadra(F.col("v1") + F.col("tax") + F.col("sc") - F.col("pag")).alias("A_actual"),
    cuadra(F.col("v2") + F.col("tax") + F.col("sc") - F.col("pag")).alias("B_por_cantidad"),
    cuadra(F.col("v1") - F.col("desc") + F.col("tax") + F.col("sc") - F.col("pag")).alias("C_menos_desc"),
    cuadra(F.col("v1") + F.col("desc") + F.col("tax") + F.col("sc") - F.col("pag")).alias("D_mas_desc"),
    cuadra(F.col("v1") + F.col("tax") + F.col("sc") + F.col("prop") - F.col("pag")).alias("E_con_propina"),
    F.round(F.avg((F.col("n_pagos") == 0).cast("int")) * 100, 1).alias("pct_sin_pago")))

## 10. Ventas por canal
Agrupamos tickets y ventas por canal y medimos qué porcentaje no registra comensales.

In [0]:
print("copias exactas:", r.count() - r.dropDuplicates().count())

rc = spark.table("workspace.default.revenue_centers").withColumnRenamed("Id", "rc_id").withColumnRenamed("Name", "canal")
c = spark.table("workspace.default.checks_resumen")
display(c.join(rc, c.RevenueCenterId == rc.rc_id, "left").groupBy("canal").agg(
    F.count("*").alias("tickets"),
    F.round(F.sum("ventas"), 0).alias("ventas"),
    F.round(F.avg((F.col("TrafficCount") == 0).cast("int")) * 100, 1).alias("pct_sin_comensales")
).orderBy(F.desc("ventas")))

In [0]:
dif = F.col("pag") - (F.col("v1") - F.col("desc") + F.col("tax") + F.col("sc"))
display(d.select(dif.alias("dif")).summary("min", "5%", "25%", "50%", "75%", "95%", "max"))

In [0]:
dd = d.withColumn("dif", dif).filter(F.abs("dif") > 0.05)
display(dd.select(
    F.round(F.avg((F.col("dif") > 0).cast("int")) * 100, 1).alias("pct_pago_de_mas"),
    F.round(F.avg((F.abs(F.col("dif") - F.round("dif")) < 0.01).cast("int")) * 100, 1).alias("pct_monto_redondo")))

# 11. Base para el análisis
Quitamos duplicados y definimos venta neta = venta − descuentos.

In [0]:
loc = spark.table("workspace.default.locations").select(F.col("Id").alias("loc_id"), F.col("Name").alias("sucursal"))
c = (spark.table("workspace.default.checks_resumen")
     .dropDuplicates(["LocationId", "BusinessDate", "Id"])
     .withColumn("venta_neta", F.col("ventas") - F.col("descuentos")))

## 12. Análisis de negocio
Tendencia diaria, patrón por día de la semana y hora, desempeño por sucursal y productos más vendidos (sin contar modificadores).

Tendencia diaria (gráfica de línea, dt en X

In [0]:
display(c.groupBy("dt").agg(F.round(F.sum("venta_neta")).alias("venta_neta"), F.count("*").alias("tickets")).orderBy("dt"))

Databricks visualization. Run in Databricks to view.

Día de la semana y hora

In [0]:
display(c.groupBy(F.dayofweek("dt").alias("n"), "dia_semana")
         .agg(F.round(F.sum("venta_neta") / F.countDistinct("dt")).alias("venta_prom_por_dia")).orderBy("n"))
display(c.groupBy("hora").agg(F.count("*").alias("tickets")).orderBy("hora"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Sucursales

In [0]:
display(c.join(loc, c.LocationId == loc.loc_id).groupBy("sucursal").agg(
    F.round(F.sum("venta_neta")).alias("venta_neta"),
    F.round(F.sum("venta_neta") / F.count("*"), 2).alias("ticket_prom"),
    F.round(F.sum(F.when(F.col("TrafficCount") > 0, F.col("venta_neta"))) / F.sum("TrafficCount"), 2).alias("por_comensal")
).orderBy(F.desc("venta_neta")).limit(10))

Databricks visualization. Run in Databricks to view.

**Productos** (solo platillos principales, sin modificadores)

In [0]:
it = (spark.table("workspace.default.checks").select(F.explode("ItemsSold").alias("x")).select("x.*")
      .filter("ParentSequence = 0"))
display(it.groupBy("ItemName", "ItemSubCatName").agg(F.sum("Quantity").alias("unidades"), F.round(F.sum("SoldPrice")).alias("venta"))
          .orderBy(F.desc("venta")).limit(15))
display(it.groupBy("ItemMajorCatName").agg(F.round(F.sum("SoldPrice")).alias("venta")).orderBy(F.desc("venta")))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F
r = spark.table("workspace.default.checks")
s = lambda arr, campo: F.aggregate(F.coalesce(arr, F.array()), F.lit(0.0), lambda a, x: a + F.coalesce(x[campo].cast("double"), F.lit(0.0)))
x = (r.withColumn("dif", s("Payments", "Amount") - (s("ItemsSold", "SoldPrice") - s("Discounts", "Amount") + s("Tax", "Amount") + F.coalesce("ServiceCharge", F.lit(0.0))))
      .withColumn("con_efectivo", F.exists(F.coalesce("Payments", F.array()), lambda p: p.PayCatName == "CASH")))
display(x.groupBy("con_efectivo").agg(F.count("*").alias("tickets"),
        F.round(F.avg((F.col("dif") > 0.05).cast("int")) * 100, 1).alias("pct_pago_de_mas")))

In [0]:
total = r.count()
for k in [["LocationId", "BusinessDate", "Id"],
          ["LocationId", "BusinessDate", "Id", "RevenueCenterId"],
          ["LocationId", "BusinessDate", "Id", "OpenTime"],
          ["LocationId", "BusinessDate", "Id", "OpenTime", "TableName"]]:
    print(k, "→ repetidos:", total - r.select(*k).distinct().count())
print("copias exactas:", total - r.dropDuplicates().count())

In [0]:
display(r.select(F.round(F.avg((s("Payments", "Gratuity") > 0).cast("int")) * 100, 2).alias("pct_con_gratuity")))

In [0]:
c = (spark.table("workspace.default.checks_resumen")
     .dropDuplicates(["LocationId", "BusinessDate", "Id"])
     .withColumn("venta_neta", F.col("ventas") - F.col("descuentos")))
q1, q3 = c.approxQuantile("venta_neta", [0.25, 0.75], 0.001)
lim = q3 + 1.5 * (q3 - q1)
n_out = c.filter(F.col("venta_neta") > lim).count()
print(f"Q1={q1:.2f}  Q3={q3:.2f}  límite={lim:.2f}  outliers={n_out} ({n_out / c.count() * 100:.1f}%)  máximo={c.agg(F.max('venta_neta')).first()[0]:.2f}")

In [0]:
r = spark.table("workspace.default.checks")
s = lambda arr, campo: F.aggregate(F.coalesce(arr, F.array()), F.lit(0.0), lambda a, x: a + F.coalesce(x[campo].cast("double"), F.lit(0.0)))
ej = (r.select("LocationId", "BusinessDate", "Id",
        s("ItemsSold", "SoldPrice").alias("venta"),
        s("Discounts", "Amount").alias("descuento"),
        s("Tax", "Amount").alias("impuestos"),
        F.col("ServiceCharge").alias("cargo_servicio"),
        s("Payments", "Amount").alias("pagado"),
        s("Payments", "Gratuity").alias("gratuity"),
        F.transform("Payments", lambda p: p.PayCatName).alias("forma_pago"))
      .withColumn("total", F.col("venta") - F.col("descuento") + F.col("impuestos") + F.col("cargo_servicio"))
      .withColumn("dif", F.round(F.col("pagado") - F.col("total"), 2))
      .filter(F.abs("dif") > 0.05))
display(ej.limit(6))

In [0]:
r = spark.table("workspace.default.checks")
s = lambda arr, campo: F.aggregate(F.coalesce(arr, F.array()), F.lit(0.0), lambda a, x: a + F.coalesce(x[campo].cast("double"), F.lit(0.0)))
t = (r.select(
        s("ItemsSold", "SoldPrice").alias("venta"), s("Discounts", "Amount").alias("desc"),
        s("Tax", "Amount").alias("tax"), F.coalesce("ServiceCharge", F.lit(0.0)).alias("sc"),
        s("Payments", "Amount").alias("pag"), s("Payments", "Gratuity").alias("grat"))
     .withColumn("dif", F.round(F.col("pag") + F.col("grat") - (F.col("venta") - F.col("desc") + F.col("tax") + F.col("sc")), 2)))
display(t.select(
    F.round(F.avg((F.abs("dif") <= 0.05).cast("int")) * 100, 2).alias("pct_cuadra"),
    F.sum((F.col("dif") > 0.05).cast("int")).alias("pago_de_mas"),
    F.sum((F.col("dif") < -0.05).cast("int")).alias("pago_de_menos"),
    F.round(F.avg((F.col("grat") < 0).cast("int")) * 100, 1).alias("pct_con_propina")))

In [0]:
display(spark.table("workspace.default.checks")
        .select(F.explode("Discounts").alias("d"))
        .groupBy("d.DiscountName").agg(F.count("*").alias("veces"), F.round(F.sum("d.Amount"), 0).alias("monto"))
        .orderBy(F.desc("veces")))

display(spark.table("workspace.default.checks")
        .select(F.explode("Voids").alias("v"))
        .groupBy("v.VoidCategoryName").agg(F.count("*").alias("veces"), F.round(F.sum("v.Amount"), 0).alias("monto"))
        .orderBy(F.desc("veces")))

In [0]:
KEY = ["LocationId", "BusinessDate", "Id"]
r = spark.table("workspace.default.checks").dropDuplicates(KEY)

it = r.select(*KEY, F.explode("ItemsSold").alias("i")).select(*KEY, F.col("i.EmployeeId").alias("EmployeeId"))
multi = it.groupBy(*KEY).agg(F.countDistinct("EmployeeId").alias("n_emp"))
display(multi.select(F.round(F.avg((F.col("n_emp") > 1).cast("int")) * 100, 2).alias("pct_tickets_varios_empleados")))

srv = r.select(*KEY, F.coalesce(F.try_element_at("Payments", F.lit(1))["EmployeeId"],
                                F.try_element_at("ItemsSold", F.lit(1))["EmployeeId"]).alias("server_id"))
vo = r.select(*KEY, F.explode("Voids").alias("v")).select(*KEY, F.col("v.EmployeeId").alias("EmployeeId"))
vj = vo.join(srv, KEY, "left")
display(vj.select(F.round(F.avg((F.col("EmployeeId") != F.col("server_id")).cast("int")) * 100, 2).alias("pct_voids_de_otro_empleado")))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.horoz
COMMENT 'Tablas del pipeline de fraude de Team Horoz (bronze, silver, gold)';

In [0]:
def listar(p):
    out = []
    for f in dbutils.fs.ls(p):
        out += listar(f.path) if f.isDir() else [f]
    return out

fs = listar("/Volumes/workspace/default/applebees/Applebees_Challenge/facts/checks")
tam = sorted(f.size for f in fs)
print(len(fs), "archivos | total:", round(sum(tam) / 1e9, 2), "GB",
      "| mediana:", round(tam[len(tam) // 2] / 1e6, 1), "MB",
      "| más grande:", round(tam[-1] / 1e6, 1), "MB")
print("ejemplo:", fs[0].path)